In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import pickle
import sbi.utils as utils
import torch
import seaborn as sns
from seaborn import kdeplot
import numpy as np
import sys  
sys.path.insert(1, '../')
from collective_posterior import CollectivePosterior
from simulators import WF, wrapper_hierarchical

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400

import warnings
warnings.simplefilter('ignore', Warning)

generation = pd.read_csv('empirical_data/Chuong_116_gens.txt').columns

import matplotlib
def change_font(fontsize):
    matplotlib.rcParams['xtick.labelsize'] = fontsize
    matplotlib.rcParams['ytick.labelsize'] = fontsize
    matplotlib.rcParams['font.size'] = fontsize


In [2]:
# Prior doesn't have to be identical to network's prior (here it is)
prior_min = np.log10(np.array([1e-2,1e-7,1e-8]))
prior_max = np.log10(np.array([1,1e-2,1e-2]))
prior = utils.BoxUniform(low=torch.tensor(prior_min), 
                         high=torch.tensor(prior_max))
posterior_chuong = pickle.load(open(f'posteriors/posterior_WF_30000_20.pkl', 'rb'))

collective_posteriors = {}
epsilons = {'wt': -300, 'ltr': -150, 'ars': -10, 'all': -10}

lines = ['wt','ltr','ars','all']
for line in lines:
    amortized_posterior = posterior_chuong
    Xs = pd.read_csv(f'empirical_data/{line}.csv', index_col=0) # observations
    Xs = torch.tensor(np.array(Xs), dtype=torch.float32)
    log_C = 1
    op = CollectivePosterior(prior, Xs, amortized_posterior,log_C, epsilon=-100)
    op.get_log_C()
    collective_posteriors[line] = op # to use throughout the notebook
    op.sample(50, method='unimodal')
    
rep_colors = {'wt':"black", 'ltr':"#6699cc", 'ars': "#e26d5c", 'all':"#DEBD52", "lauer": "grey"}
label_dict = {'wt': '(A) Wildtype', 'ltr': '(B) LTRΔ', 'ars': '(C) ARSΔ', 'all': '(D) ALLΔ', 'lauer': 'Lauer'}

Sampling: 100%|██████████| 50/50 [00:00<00:00, 373.81it/s]


In [3]:
# plot
torch.manual_seed(137)
np.random.seed(137)
change_font(26)
fig, ax = plt.subplots(4,1, figsize=(18,36))
fig.tight_layout(pad=10)

ax[0].set_xlabel('Time')
# ax[0].set_ylabel('Frequency')
ax[1].set_xlabel('θ')
ax[2].set_xlabel('θ')
ax[3].set_xlabel('Time')
# ax[3].set_ylabel('Frequency')

ax[0].set_title('Empirical Data Samples $x_1,...,x_n$')
ax[1].set_title('Individual Posterior Distributions\n$p(θ|x_i)$')
ax[2].set_title('Collective Posterior Distribution\n$p(θ|x_1,...,x_n)$')
ax[3].set_title('Predictive Checks')


# Remove frames and ticks
# for i in range(len(ax)):
#     ax[i].set(frame_on=False)
#     ax[i].set_xticks([])
#     ax[i].set_yticks([])

line = 'ltr'
cp = collective_posteriors[line]
color='grey'

# first subfig - data
x_outlier = collective_posteriors['wt'].Xs[2].reshape(1,-1)
data = torch.cat([cp.Xs, x_outlier])
op = CollectivePosterior(prior, data, posterior_chuong, log_C=1, epsilon=-150)
op.get_log_C()
op.sample(50)

for i in range(len(data)):
    ax[0].plot(generation, data[i], color=color, lw=1.5)

# Second subfig - posteriors
ap = posterior_chuong
for x in data:
    samples = ap.set_default_x(x).sample((100,))[:,0]
    kdeplot(data=samples, ax=ax[1], color=color, alpha=0.1, fill=True)
    # Third subfig - adding the collective posterior
    kdeplot(data=samples, ax=ax[2], color=color, alpha=0.1, fill=True)   
kdeplot(data=op.samples[:,0], ax=ax[2], color='tomato', alpha=0.5, fill=True)

# Fourth subfig - plot against data
for i in range(len(op.samples)):
    y = WF(op.samples[i], seed=137)
    plt.plot(generation,y, color='tomato', lw=0.5)
for i in range(len(data)):
    ax[3].plot(generation, data[i], color=color, lw=1.5)

    



Rejection Sampling: 54it [00:01, 27.87it/s]                        


Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

Drawing 100 posterior samples:   0%|          | 0/100 [00:00<?, ?it/s]

In [27]:
fig.savefig('collective_flow.png')